In [0]:
spark.sql("""
CREATE OR REPLACE TABLE de_mini_project.gold.fact_sales AS
WITH sales_enriched AS (
    SELECT 
        s.transaction_id,
        s.date,
        s.product_id,
        s.customer_id,
        s.store_id,
        s.sold_qty,
        s.unit_price AS actual_sold_price,
        p.selling_price AS original_marked_price,
        (s.sold_qty * s.unit_price) AS total_revenue,
        (p.selling_price - s.unit_price) * s.sold_qty AS discount_loss_amount,
        CASE WHEN r.transaction_id IS NOT NULL THEN 1 ELSE 0 END AS is_returned
    FROM de_mini_project.silver.sales s
    LEFT JOIN de_mini_project.gold.dim_product p ON s.product_id = p.product_id
    LEFT JOIN de_mini_project.silver.returns r ON s.transaction_id = r.transaction_id
)
SELECT * FROM sales_enriched
""")

In [0]:
display(spark.sql("""
    SELECT transaction_id, actual_sold_price, original_marked_price, discount_loss_amount 
    FROM de_mini_project.gold.fact_sales 
    WHERE discount_loss_amount > 0 
    LIMIT 5
"""))